# 📘 Capstone Project: IoT/Manufacturing Data Platform
## Build a Smart Factory Analytics Pipeline

**Scenario:** You are building the data platform for a smart factory with 100 sensors
across 30 machines generating 200K+ readings.

**Deliverables:**
1. High-volume sensor ingestion (Bronze)
2. Anomaly detection (statistical + pattern-based)
3. Machine health scoring (composite 0-100)
4. Quality monitoring dashboard (pass/fail by shift/machine)
5. Predictive maintenance indicators
6. Alerting system

**Estimated time:** 6-8 hours

**Prerequisites:** Notebook 06 (iot_manufacturing_dw must exist)

---

---
# 🏗️ Section 1: Architecture Design

```
┌──────────────────────────────────────────────────────────────────────┐
│              SMART FACTORY DATA PLATFORM                               │
├──────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  SENSORS (100)        BRONZE              SILVER              GOLD    │
│  ──────────────       ──────              ──────              ────    │
│  200K readings ──▶ bronze_readings ──▶ silver_readings ──▶ gold_health│
│  (4 types)        (partitioned by     (anomaly flagged)   (machine   │
│                    event_date)         (deduplicated)       scores)   │
│                                                                        │
│  MACHINES (30)                                                         │
│  50K events ─────▶ bronze_events ────▶ silver_events ───▶ gold_quality│
│                                                                        │
│  QUALITY (20K)                                                         │
│  checks ─────────▶ bronze_quality ───▶ silver_quality ──▶ gold_maint │
│                                                                        │
│  OPTIMIZATION: Liquid Clustering on (sensor_id, reading_timestamp)    │
│  RETENTION: Raw 90 days, Aggregated 2 years                           │
│  ALERTING: Health < 50 → CRITICAL, Anomaly burst → IMMEDIATE         │
└──────────────────────────────────────────────────────────────────────┘
```

---
# 🥉 Section 2: Sensor Data Ingestion (Bronze)

In [ ]:
from pyspark.sql.functions import *
import time

spark.sql("USE iot_manufacturing_dw")

# Ingest 200K+ sensor readings into Bronze
start = time.time()
readings = spark.table("sensor_readings")
bronze_readings = (readings
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source", lit("iot_gateway"))
    .withColumn("_batch_id", lit("capstone_iot_001"))
    .withColumn("event_date", to_date("reading_timestamp"))
)

# Write partitioned by date (time-series optimization)
bronze_readings.write.format("delta").mode("overwrite").partitionBy("event_date").saveAsTable("capstone_bronze_readings")

duration = time.time() - start
count = spark.table("capstone_bronze_readings").count()
print(f"✅ Bronze readings: {count:,} rows in {duration:.1f}s ({count/duration:,.0f} rows/sec)")

In [ ]:
# Bronze for production events and quality checks
events = spark.table("production_events")
bronze_events = events.withColumn("_ingested_at", current_timestamp()).withColumn("_batch_id", lit("capstone_iot_001"))
bronze_events.write.format("delta").mode("overwrite").saveAsTable("capstone_bronze_events")

quality = spark.table("quality_checks")
bronze_quality = quality.withColumn("_ingested_at", current_timestamp()).withColumn("_batch_id", lit("capstone_iot_001"))
bronze_quality.write.format("delta").mode("overwrite").saveAsTable("capstone_bronze_quality")

print(f"✅ Bronze events: {spark.table('capstone_bronze_events').count():,}")
print(f"✅ Bronze quality: {spark.table('capstone_bronze_quality').count():,}")

---
# 🔍 Section 3: Anomaly Detection (Silver)

In [ ]:
# Statistical anomaly detection using window functions
from pyspark.sql.functions import *
from pyspark.sql.window import Window

bronze = spark.table("capstone_bronze_readings")

# Rolling baseline per sensor (last 100 readings)
w_baseline = Window.partitionBy("sensor_id").orderBy("reading_timestamp").rowsBetween(-100, -1)

silver_readings = (bronze
    # Deduplicate (same sensor + timestamp)
    .dropDuplicates(["sensor_id", "reading_timestamp"])
    # Filter invalid
    .filter("sensor_id IS NOT NULL AND value IS NOT NULL AND quality_flag = 'valid'")
    # Calculate rolling stats
    .withColumn("rolling_mean", avg("value").over(w_baseline))
    .withColumn("rolling_std", stddev("value").over(w_baseline))
    # Z-score anomaly detection
    .withColumn("z_score", 
        when(col("rolling_std") > 0, 
            abs((col("value") - col("rolling_mean")) / col("rolling_std")))
        .otherwise(0))
    # Flag anomalies
    .withColumn("is_statistical_anomaly", col("z_score") > 3.0)
    # Sudden jump detection (> 50% change from previous)
    .withColumn("prev_value", lag("value").over(Window.partitionBy("sensor_id").orderBy("reading_timestamp")))
    .withColumn("pct_change", 
        when(col("prev_value") > 0, abs(col("value") - col("prev_value")) / col("prev_value"))
        .otherwise(0))
    .withColumn("is_jump_anomaly", col("pct_change") > 0.5)
    # Combined anomaly flag
    .withColumn("is_anomaly_detected", col("is_statistical_anomaly") | col("is_jump_anomaly"))
    .withColumn("anomaly_severity",
        when(col("z_score") > 5, "critical")
        .when(col("z_score") > 3, "warning")
        .otherwise("normal"))
    .withColumn("_processed_at", current_timestamp())
)

silver_readings.write.format("delta").mode("overwrite").saveAsTable("capstone_silver_readings")
total = spark.table("capstone_silver_readings").count()
anomalies = spark.table("capstone_silver_readings").filter("is_anomaly_detected = true").count()
print(f"✅ Silver readings: {total:,} rows")
print(f"   Anomalies detected: {anomalies:,} ({anomalies/total*100:.2f}%)")

In [ ]:
# ============================================
# 🎯 YOUR TURN: Anomaly Summary by Sensor Type
# ============================================
# Join silver_readings with sensors table to get sensor_type
# Group by sensor_type and anomaly_severity
# Calculate: count, avg_z_score, max_z_score
# Which sensor type has the most anomalies?
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
silver = spark.table("capstone_silver_readings")
sensors = spark.table("sensors")

anomaly_summary = (silver
    .filter("is_anomaly_detected = true")
    .join(sensors.select("sensor_id", "sensor_type"), "sensor_id")
    .groupBy("sensor_type", "anomaly_severity")
    .agg(
        count("*").alias("anomaly_count"),
        round(avg("z_score"), 2).alias("avg_z_score"),
        round(max("z_score"), 2).alias("max_z_score")
    )
    .orderBy("anomaly_count", ascending=False)
)
anomaly_summary.show()

---
# 💚 Section 4: Machine Health Scoring (Gold)

In [ ]:
# Composite health score per machine (0-100)
from pyspark.sql.functions import *

silver = spark.table("capstone_silver_readings")
sensors = spark.table("sensors")
events = spark.table("capstone_bronze_events")
quality = spark.table("capstone_bronze_quality")

# Component 1: Anomaly frequency (lower = healthier)
anomaly_rate = (silver
    .join(sensors.select("sensor_id", "machine_id"), "sensor_id")
    .groupBy("machine_id")
    .agg(
        count("*").alias("total_readings"),
        sum(when(col("is_anomaly_detected"), 1).otherwise(0)).alias("anomaly_count")
    )
    .withColumn("anomaly_pct", col("anomaly_count") / col("total_readings") * 100)
    .withColumn("anomaly_score", greatest(lit(0), lit(100) - col("anomaly_pct") * 10))
)

# Component 2: Quality pass rate
quality_rate = (quality
    .groupBy("machine_id")
    .agg(
        count("*").alias("total_checks"),
        sum(when(col("pass_fail") == "pass", 1).otherwise(0)).alias("passed")
    )
    .withColumn("quality_score", col("passed") / col("total_checks") * 100)
)

# Component 3: Error event rate
error_rate = (events
    .groupBy("machine_id")
    .agg(
        count("*").alias("total_events"),
        sum(when(col("event_type") == "error", 1).otherwise(0)).alias("errors")
    )
    .withColumn("reliability_score", greatest(lit(0), lit(100) - col("errors") / col("total_events") * 200))
)

# Combine into composite score
machines = spark.table("machines")
health_scores = (machines
    .select("machine_id", "machine_name", "machine_type", "production_line")
    .join(anomaly_rate.select("machine_id", "anomaly_score"), "machine_id", "left")
    .join(quality_rate.select("machine_id", "quality_score"), "machine_id", "left")
    .join(error_rate.select("machine_id", "reliability_score"), "machine_id", "left")
    .na.fill(50)
    .withColumn("health_score", round(
        col("anomaly_score") * 0.35 +
        col("quality_score") * 0.35 +
        col("reliability_score") * 0.30, 1))
    .withColumn("health_status",
        when(col("health_score") >= 70, "healthy")
        .when(col("health_score") >= 50, "warning")
        .otherwise("critical"))
    .withColumn("_computed_at", current_timestamp())
)

health_scores.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_machine_health")
print(f"✅ Machine health scores: {spark.table('capstone_gold_machine_health').count()} machines")
health_scores.orderBy("health_score").show(10)

---
# 📊 Section 5: Quality Monitoring Dashboard

In [ ]:
# Quality by shift, machine, and time
from pyspark.sql.functions import *

quality = spark.table("capstone_bronze_quality")

gold_quality = (quality
    .withColumn("check_date", to_date("check_timestamp"))
    .withColumn("check_hour", hour("check_timestamp"))
    .withColumn("shift",
        when(col("check_hour").between(6, 13), "morning")
        .when(col("check_hour").between(14, 21), "afternoon")
        .otherwise("night"))
    .groupBy("check_date", "shift", "machine_id")
    .agg(
        count("*").alias("total_checks"),
        sum(when(col("pass_fail") == "pass", 1).otherwise(0)).alias("passed"),
        sum(when(col("pass_fail") == "fail", 1).otherwise(0)).alias("failed"),
        round(avg("measurement"), 4).alias("avg_measurement")
    )
    .withColumn("pass_rate", round(col("passed") / col("total_checks") * 100, 2))
    .withColumn("_computed_at", current_timestamp())
)

gold_quality.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_quality_dashboard")
print(f"✅ Quality dashboard: {spark.table('capstone_gold_quality_dashboard').count():,} rows")

# Worst performing shifts
gold_quality.orderBy("pass_rate").show(5)

---
# 🔧 Section 6: Predictive Maintenance Indicators

In [ ]:
# Degradation detection: trending health scores
from pyspark.sql.functions import *
from pyspark.sql.window import Window

health = spark.table("capstone_gold_machine_health")

# Simple maintenance priority ranking
maintenance_priority = (health
    .withColumn("priority_rank", 
        when(col("health_status") == "critical", 1)
        .when(col("health_status") == "warning", 2)
        .otherwise(3))
    .withColumn("maintenance_urgency",
        when(col("health_score") < 40, "IMMEDIATE")
        .when(col("health_score") < 60, "This week")
        .when(col("health_score") < 75, "This month")
        .otherwise("Scheduled"))
    .orderBy("health_score")
)

maintenance_priority.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_maintenance_priority")
print("✅ Maintenance priority:")
maintenance_priority.select("machine_id", "machine_name", "health_score", "health_status", "maintenance_urgency").show(10)

---
# 🔔 Section 7: Alerting System

In [ ]:
import pandas as pd
from datetime import datetime

# Generate alerts based on health scores and anomaly rates
health = spark.table("capstone_gold_machine_health").collect()
alerts = []

for machine in health:
    if machine.health_score < 50:
        alerts.append({
            "alert_type": "machine_health_critical",
            "machine_id": machine.machine_id,
            "severity": "critical",
            "message": f"{machine.machine_name} health score {machine.health_score} (CRITICAL)",
            "generated_at": datetime.now().isoformat()
        })
    elif machine.health_score < 70:
        alerts.append({
            "alert_type": "machine_health_warning",
            "machine_id": machine.machine_id,
            "severity": "warning",
            "message": f"{machine.machine_name} health score {machine.health_score} (WARNING)",
            "generated_at": datetime.now().isoformat()
        })

if alerts:
    alerts_df = spark.createDataFrame(pd.DataFrame(alerts))
    alerts_df.write.format("delta").mode("overwrite").saveAsTable("capstone_iot_alerts")
    print(f"🔔 Generated {len(alerts)} alerts:")
    for a in alerts[:5]:
        icon = "🔴" if a["severity"] == "critical" else "🟡"
        print(f"  {icon} {a['message']}")
else:
    print("  No alerts generated")

---
# ✅ Section 8: Final Verification

In [ ]:
%sql
USE iot_manufacturing_dw;
SELECT 'capstone_bronze_readings' AS tbl, COUNT(*) AS rows FROM capstone_bronze_readings
UNION ALL SELECT 'capstone_bronze_events', COUNT(*) FROM capstone_bronze_events
UNION ALL SELECT 'capstone_bronze_quality', COUNT(*) FROM capstone_bronze_quality
UNION ALL SELECT 'capstone_silver_readings', COUNT(*) FROM capstone_silver_readings
UNION ALL SELECT 'capstone_gold_machine_health', COUNT(*) FROM capstone_gold_machine_health
UNION ALL SELECT 'capstone_gold_quality_dashboard', COUNT(*) FROM capstone_gold_quality_dashboard
UNION ALL SELECT 'capstone_gold_maintenance_priority', COUNT(*) FROM capstone_gold_maintenance_priority
ORDER BY tbl;

---
# 📗 Enhancement: IoT Pipeline -- Production Patterns

## Architecture Overview

```
Source Systems
    |
    v
Bronze Layer (raw, append-only)
    |
    v
Silver Layer (cleaned, validated, deduplicated)
    |
    v
Gold Layer (aggregated, business-ready)
    |
    v
Serving (SQL Warehouse, APIs, ML)
```

## Key Tables: sensor_readings, devices, alerts, maintenance_logs

## Production Checklist

| Item | Status |
|------|--------|
| Idempotent pipeline (safe to rerun) | Required |
| Data quality checks at each layer | Required |
| Incremental processing (not full refresh) | Required |
| Error handling and dead letter queue | Required |
| Monitoring and alerting | Required |
| Documentation and data catalog | Required |

## Common Patterns Used

1. **Auto Loader** for file ingestion
2. **MERGE** for CDC and upserts
3. **ROW_NUMBER** for deduplication
4. **Window functions** for running totals, rankings
5. **DLT expectations** for quality gates
6. **Lakeflow Jobs** for orchestration

In [ ]:
# IoT Pipeline -- Key SQL Patterns
print("IoT Pipeline -- Production SQL Patterns")
print("=" * 60)

# Pattern 1: Incremental load with watermark
print("\n1. Incremental load pattern:")
print("""
    MERGE INTO silver.sensor_readings AS target
    USING (
        SELECT * FROM bronze.raw_sensor_readings
        WHERE _ingested_at > (SELECT MAX(_ingested_at) FROM silver.sensor_readings)
    ) AS source
    ON target.id = source.id
    WHEN MATCHED AND source._row_hash != target._row_hash THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

# Pattern 2: Deduplication
print("2. Deduplication pattern:")
print("""
    SELECT * FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY id
                ORDER BY _ingested_at DESC
            ) AS rn
        FROM bronze.raw_sensor_readings
    ) WHERE rn = 1
""")

# Pattern 3: Quality checks
print("3. Quality check pattern:")
quality_checks = [
    f"SELECT COUNT(*) FROM silver.sensor_readings WHERE id IS NULL",
    f"SELECT COUNT(*) - COUNT(DISTINCT id) FROM silver.sensor_readings",
    f"SELECT COUNT(*) FROM silver.sensor_readings WHERE amount < 0",
]
for qc in quality_checks:
    print(f"   {qc}")


---
# Enhancement: IoT Pipeline -- Production Patterns

## Architecture

```
Source: sensor_readings, devices, alerts
    |
    v (Auto Loader / Kafka)
Bronze Layer (raw, append-only, Delta)
    |
    v (DLT with expectations)
Silver Layer (cleaned, validated, typed)
    |
    v (Spark / dbt)
Gold Layer (aggregated, business-ready)
    |
    v
Serving (SQL Warehouse, APIs, ML)
```

## Key Tables: sensor_readings, devices, alerts

## Production Checklist

| Item | Requirement |
|------|-------------|
| Idempotent pipeline | Safe to rerun for any date |
| Quality checks | At each layer boundary |
| Incremental processing | Not full refresh |
| Error handling | DLQ for bad records |
| Monitoring | Alerts on failure |
| Documentation | Data catalog entries |

## Common Patterns Used

1. **Auto Loader** for file ingestion (cloudFiles format)
2. **MERGE** for CDC and upserts
3. **ROW_NUMBER** for deduplication
4. **Window functions** for running totals, rankings
5. **DLT expectations** for quality gates
6. **Lakeflow Jobs** for orchestration

In [ ]:
# IoT Pipeline -- Key SQL Patterns
print("IoT Pipeline -- Production SQL Patterns")
print("=" * 60)

patterns = {
    "Incremental load (watermark)": f"""
    MERGE INTO silver.sensor_readings AS target
    USING (
        SELECT * FROM bronze.raw_sensor_readings
        WHERE _ingested_at > (SELECT COALESCE(MAX(_ingested_at), '1900-01-01')
                              FROM silver.sensor_readings)
    ) AS source
    ON target.id = source.id
    WHEN MATCHED AND source._row_hash != target._row_hash THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """,

    "Deduplication (keep latest)": f"""
    SELECT * FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY id
                ORDER BY _ingested_at DESC
            ) AS rn
        FROM bronze.raw_sensor_readings
    ) WHERE rn = 1
    """,

    "Quality checks": f"""
    SELECT 'null_id' AS check_name, COUNT(*) AS failures
    FROM silver.sensor_readings WHERE id IS NULL
    UNION ALL
    SELECT 'duplicate_id', COUNT(*) - COUNT(DISTINCT id)
    FROM silver.sensor_readings
    UNION ALL
    SELECT 'negative_amount', COUNT(*)
    FROM silver.sensor_readings WHERE amount < 0
    """,
}

for pname, sql in patterns.items():
    print(f"\n" + pname + ":")
    print(sql)


---
# 🎓 Capstone Complete!

**What was built:**
- High-volume Bronze ingestion (200K+ readings, partitioned by date)
- Statistical anomaly detection (z-score, jump detection, severity levels)
- Composite machine health scoring (anomaly + quality + reliability)
- Quality monitoring dashboard (by shift, machine, time)
- Predictive maintenance priority ranking
- Alerting system (health threshold-based)

**IoT-specific patterns used:**
- Time-series partitioning (by event_date)
- Rolling window calculations (baseline per sensor)
- High-volume optimization (coalesce, partition pruning)
- Multi-signal composite scoring

**Next:** Notebook 24 — Streaming Processing

---